# E1: BoW + MultinomialNB, tối ưu trên bộ train / val / test

**Mục tiêu:** đẩy Toxic F1 của BoW lên cao nhất có thể mà vẫn giữ nguyên thuật toán của E1.

**Đã tối ưu:** tách từ hay không, chuẩn hóa ký tự kéo dài, n-gram từ, n-gram ký tự, đếm hay nhị phân, `min_df`, chọn đặc trưng chi2, tham số thuật toán và ngưỡng.

**Giữ kỷ luật:**
- Đầu vào chỉ là `message_clean`. Không dùng `message_normalized`, vì cột này do AI gán nhãn tạo ra và gần như lộ nhãn.
- Tham số và ngưỡng chỉ chọn trên Train và Validation; Test chỉ chấm một lần ở cuối.

**Lưu ý khi đọc kết quả:**
- 3 file được chia theo từng bình luận, nên nhiều bài đăng nằm ở cả Train lẫn Val/Test. Mục "Theo bài đăng" tách riêng phần này.
- Tỷ lệ toxic khoảng 21,7%, cao hơn phân bố tự nhiên, và nhãn theo chính sách Keyword-Priority v2.2.

## Chạy trên Google Colab
1. Vào [colab.research.google.com](https://colab.research.google.com), chọn **Tải lên** và mở notebook này.
2. Chọn **Thời gian chạy → Chạy tất cả**. Không cần GPU.
3. Khi có nút **Chọn tệp**, chọn cùng lúc `train.xlsx`, `val.xlsx`, `test.xlsx`.
4. Cuối notebook sẽ tải về file zip chứa mô hình, bảng điểm và biểu đồ.

Dữ liệu có nội dung bình luận và tên thật, nên chỉ tải lên tài khoản của bạn và đừng chia sẻ notebook kèm kết quả.

**Nếu đổi sang dữ liệu khác:** đặt `RUN_TEST = False` để chọn cấu hình bằng Validation trước, rồi mới bật lại.

## 1. Cài đặt

In [ ]:
%pip install -q scikit-learn pandas openpyxl matplotlib joblib py_vncorenlp

import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # VnCoreNLP cần Java và biến JAVA_HOME để tách từ
    if shutil.which("java") is None:
        !apt-get -qq install -y default-jre-headless
    os.environ["JAVA_HOME"] = str(Path(os.path.realpath(shutil.which("java"))).parents[1])
    print("JAVA_HOME =", os.environ["JAVA_HOME"])

## 2. Cấu hình

In [ ]:
import json
import os
import re
import unicodedata
import urllib.request
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline

EXPERIMENT = "E1"
SEED = 42
RUN_TEST = True  # Chỉ bật sau khi đã chốt cấu hình bằng Validation; không chỉnh tham số theo Test

BASE = Path.cwd().resolve()
CACHE = BASE / "data_splits_chuan"
VNCORE_DIR = BASE / "vncorenlp"
OUT = BASE / "results_chuan" / EXPERIMENT

for folder in [CACHE, VNCORE_DIR, OUT]:
    folder.mkdir(parents=True, exist_ok=True)

## 3. Tải 3 file Excel lên

In [ ]:
UPLOAD_AGAIN = False  # Đặt True để thay bằng 3 file khác khi đã có file trong phiên làm việc
REQUIRED = ["train", "val", "test"]


def target_for(name):
    """Colab đổi tên file trùng thành "train (1).xlsx", nên nhận file theo từ khóa rồi lưu về tên chuẩn."""
    low = Path(name).name.lower()
    if low.endswith(".xlsx"):
        found = [key for key in REQUIRED if key in low]
        return BASE / f"{found[0]}.xlsx" if len(found) == 1 else None
    return None


def missing_files():
    return [f"{key}.xlsx" for key in REQUIRED if not (BASE / f"{key}.xlsx").exists()]


if IN_COLAB:
    from google.colab import files

    while UPLOAD_AGAIN or missing_files():
        print("Chọn các file Excel. Tên file cần chứa train, val hoặc test. Còn thiếu:", missing_files())
        uploaded = files.upload()
        assert uploaded, "Chưa chọn file nào. Chạy lại cell này để tải lên."
        for name, content in uploaded.items():
            target = target_for(name)
            if BASE / name != target:
                (BASE / name).unlink(missing_ok=True)
            if target is None:
                print("Bỏ qua:", name)
            else:
                target.write_bytes(content)
                print("Đã nhận", name, "->", target.name)
        UPLOAD_AGAIN = False

assert not missing_files(), f"Thiếu {missing_files()} trong {BASE}. Hãy đặt các file này cạnh notebook."
print("Đủ dữ liệu:", [f"{key}.xlsx" for key in REQUIRED])

## 4. Đọc 3 file dữ liệu

In [ ]:
COLUMNS = ["sample_id", "post_id", "comment_id", "post_message", "parent_message", "message_clean", "label"]
frames = {
    name: pd.read_excel(BASE / f"{name}.xlsx", dtype={"sample_id": str, "post_id": str, "comment_id": str})[COLUMNS]
    for name in ["train", "val", "test"]
}
for frame in frames.values():
    frame["message_clean"] = frame["message_clean"].fillna("")
    assert set(frame["label"]) == {0, 1}

ids = {name: set(frame["sample_id"]) for name, frame in frames.items()}
assert not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"]), "Trùng sample_id giữa các tập"

train_posts = set(frames["train"]["post_id"])
summary = pd.DataFrame({
    name: {
        "binh_luan": len(frame), "bai_dang": frame["post_id"].nunique(), "toxic": frame["label"].sum(),
        "ty_le_toxic": round(frame["label"].mean(), 4),
        "dong_co_bai_trong_train": frame["post_id"].isin(train_posts).sum() if name != "train" else 0,
    } for name, frame in frames.items()
}).T
summary

## 5. Tách từ bằng VnCoreNLP

In [ ]:
SEG_FILE = CACHE / "message_segmented.csv"
everything = pd.concat(frames.values(), ignore_index=True)


def load_segmenter():
    """py_vncorenlp.download_model cần wget nên lỗi trên Windows; chỉ tải đúng 3 file mà wseg cần."""
    import py_vncorenlp

    root = "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/"
    for name in ["VnCoreNLP-1.2.jar", "models/wordsegmenter/vi-vocab", "models/wordsegmenter/wordsegmenter.rdr"]:
        path = VNCORE_DIR / name
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(root + name, path)
    # VnCoreNLP đổi thư mục làm việc khi khởi tạo, phải trả lại để các đường dẫn sau không sai
    model = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORE_DIR))
    os.chdir(BASE)
    return model


def segment_text(segmenter, text):
    """VnCoreNLP làm hỏng ký tự ngoài BMP (🤣 thành 藍), nên chỉ tách phần chữ và giữ nguyên emoji."""
    pieces = []
    for part in re.split(r"([\U00010000-\U0010FFFF]+)", text):
        if re.fullmatch(r"[\U00010000-\U0010FFFF]+", part):
            pieces.append(part)
        elif part.strip():
            pieces.extend(segmenter.word_segment(part))
    return " ".join(pieces)


cached = pd.read_csv(SEG_FILE, dtype=str, keep_default_na=False) if SEG_FILE.exists() else None
usable = cached is not None and dict(zip(cached["sample_id"], cached["message_clean"])) == dict(zip(everything["sample_id"], everything["message_clean"]))
if not usable:
    segmenter = load_segmenter()
    cached = pd.DataFrame({
        "sample_id": everything["sample_id"], "message_clean": everything["message_clean"],
        "message_segmented": [segment_text(segmenter, t) for t in everything["message_clean"]],
    })
    cached.to_csv(SEG_FILE, index=False, encoding="utf-8")


# Tự kiểm tra: tách từ chỉ thêm khoảng trắng và "_"; VnCoreNLP có đổi vị trí dấu thanh nên so sau khi bỏ dấu
def bare(text):
    return re.sub(r"[\s_]", "", "".join(c for c in unicodedata.normalize("NFD", text) if not unicodedata.combining(c)))


assert all(bare(a) == bare(b) for a, b in zip(cached["message_clean"], cached["message_segmented"])), "Tách từ làm mất ký tự"

segmented = cached.set_index("sample_id")["message_segmented"]
train, val, test = (
    frames[name].assign(message_segmented=frames[name]["sample_id"].map(segmented)) for name in ["train", "val", "test"]
)
print("Đã tách từ", len(segmented), "dòng")

## 6. Hàm dùng chung

In [ ]:
# Mặc định CountVectorizer bỏ từ 1 ký tự và emoji, làm mất các tín hiệu như "đ", "m", 🖕
TOKEN_PATTERN = r"\w+|[^\w\s]"
ELONGATED = re.compile(r"([^\W\d_])\1{2,}")  # "nguuuu" thành "ngu"; không đụng chữ số như 1000

assert CountVectorizer(token_pattern=TOKEN_PATTERN).build_analyzer()("Mày ngu_vl đ 🖕") == ["mày", "ngu_vl", "đ", "🖕"]
assert ELONGATED.sub(r"\1", "nguuuu vclll 1000") == "ngu vcl 1000"


def get_text(frame, config):
    text = frame["message_segmented" if config["text"] == "segmented" else "message_clean"]
    return text.map(lambda t: ELONGATED.sub(r"\1", t)) if config["elong"] else text


def make_vectorizer(config):
    parts = []
    for feature in config["features"].split("+"):
        kind, low, high = feature.split("_")
        common = dict(ngram_range=(int(low), int(high)), binary=config["binary"], min_df=config["min_df"])
        parts.append((feature, CountVectorizer(token_pattern=TOKEN_PATTERN, **common) if kind == "word"
                      else CountVectorizer(analyzer="char_wb", **common)))
    return parts[0][1] if len(parts) == 1 else FeatureUnion(parts)


def make_model(config):
    steps = [("vec", make_vectorizer(config))]
    if config["pct"] < 100:
        steps.append(("select", SelectPercentile(chi2, percentile=config["pct"])))
    steps.append(("clf", make_classifier(config)))
    return Pipeline(steps)


def best_threshold(y, scores):
    """Ngưỡng cho Toxic F1 cao nhất; chỉ được gọi trên Validation hoặc phần gấp của Train."""
    precision, recall, thresholds = precision_recall_curve(y, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    i = int(np.argmax(f1))
    return float(thresholds[i]), float(f1[i])


def evaluate(y, scores, threshold):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "toxic_precision": precision_score(y, pred, zero_division=0),
        "toxic_recall": recall_score(y, pred, zero_division=0),
        "toxic_f1": f1_score(y, pred, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y, pred),
        "pr_auc": average_precision_score(y, scores),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def plot_results(y, scores, threshold, title):
    fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
    ConfusionMatrixDisplay.from_predictions(
        y, (scores >= threshold).astype(int), labels=[0, 1],
        display_labels=["Non-toxic", "Toxic"], cmap="Blues", colorbar=False, ax=left,
    )
    left.set_title(f"Ma trận nhầm lẫn: {title}")
    PrecisionRecallDisplay.from_predictions(y, scores, ax=right, plot_chance_level=True, name=EXPERIMENT)
    right.set_title(f"Đường Precision-Recall lớp Toxic: {title}")
    fig.tight_layout()
    fig.savefig(OUT / f"{title.lower()}_charts.png", dpi=150)
    plt.show()

## 7. Mô hình và không gian tham số

In [ ]:
MODEL_NAME = "BoW + MultinomialNB"


def make_classifier(config):
    return MultinomialNB(alpha=config["alpha"], fit_prior=config["fit_prior"])


def toxic_scores(model, texts):
    """Log-odds của lớp Toxic: 0 tương ứng xác suất 0,5."""
    log_proba = model.predict_log_proba(texts)
    return log_proba[:, 1] - log_proba[:, 0]


DEFAULT = {"text": "segmented", "elong": False, "features": "word_1_1", "binary": False, "min_df": 1, "pct": 100,
           "alpha": 0.1, "fit_prior": True}
STAGE1_CLF = {"alpha": [0.1, 0.5]}
STAGE3_CLF = {"alpha": [0.01, 0.03, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0], "fit_prior": [True, False]}

## 8. Dò tham số trên Validation

In [ ]:
def evaluate_config(config):
    model = make_model(config).fit(get_text(train, config), train["label"])
    scores = toxic_scores(model, get_text(val, config))
    threshold, f1 = best_threshold(val["label"], scores)
    return {**config, "config": config, "val_f1": f1, "threshold": threshold,
            "val_f1_nguong_0": f1_score(val["label"], (scores >= 0).astype(int), zero_division=0),
            "val_pr_auc": average_precision_score(val["label"], scores)}


def run_stage(configs):
    unique = list({json.dumps(c, sort_keys=True): c for c in configs}.values())
    rows = Parallel(n_jobs=-1)(delayed(evaluate_config)(c) for c in unique)
    result = pd.DataFrame(rows).sort_values(["val_f1", "val_pr_auc"], ascending=False).reset_index(drop=True)
    print(f"{len(unique)} cấu hình | Toxic F1 Validation tốt nhất: {result.loc[0, 'val_f1']:.4f}")
    return result


def expand(bases, grid):
    return [{**base, **change} for base in bases for change in ParameterGrid(grid)]

In [ ]:
# Dò theo 3 chặng để khỏi phải thử hàng chục nghìn tổ hợp
stage1 = run_stage(expand([DEFAULT], {
    "text": ["segmented", "clean"],
    "elong": [False, True],
    "features": ["word_1_1", "word_1_2", "word_1_3", "char_2_4", "char_2_5", "word_1_1+char_2_4", "word_1_2+char_2_4"],
    "binary": [False, True],
    **STAGE1_CLF,
}))
stage2 = run_stage(expand(stage1["config"].head(8), {"min_df": [1, 2, 3, 5], "pct": [100, 50, 25, 10, 5]}))
stage3 = run_stage(expand(stage2["config"].head(8), STAGE3_CLF))

grid = pd.concat([stage1, stage2, stage3], ignore_index=True)
grid.drop(columns="config").sort_values("val_f1", ascending=False).to_csv(OUT / "grid_search.csv", index=False, encoding="utf-8-sig")
stage3.drop(columns="config").head(10).round(4)

## 9. Chốt cấu hình bằng cross-validation theo `post_id`

In [ ]:
# Thử nhiều cấu hình trên cùng Validation dễ chọn phải cấu hình "may mắn", và Validation lại có
# nhiều bài đăng chung với Train. Vì vậy chốt trong top 10 bằng 5-fold theo post_id trên riêng Train.
def cv_scores(config):
    text, y = get_text(train, config), train["label"]
    scores = np.zeros(len(train))
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fit_idx, score_idx in folds.split(text, y, groups=train["post_id"]):
        scores[score_idx] = toxic_scores(make_model(config).fit(text.iloc[fit_idx], y.iloc[fit_idx]), text.iloc[score_idx])
    return {"cv_f1": best_threshold(y, scores)[1], "cv_pr_auc": average_precision_score(y, scores)}


top = stage3.head(10).reset_index(drop=True)
top = pd.concat([top, pd.DataFrame(Parallel(n_jobs=-1)(delayed(cv_scores)(c) for c in top["config"]))], axis=1)
top = top.sort_values(["cv_f1", "val_f1"], ascending=False).reset_index(drop=True)
top.drop(columns="config").to_csv(OUT / "top10_cv.csv", index=False, encoding="utf-8-sig")
top.drop(columns="config").round(4)

## 10. Huấn luyện cấu hình đã chốt

In [ ]:
best = top.loc[0, "config"]
THRESHOLD = float(top.loc[0, "threshold"])
print("Cấu hình chốt:", json.dumps(best, ensure_ascii=False), "| ngưỡng:", round(THRESHOLD, 4))

model = make_model(best).fit(get_text(train, best), train["label"])
val_scores = toxic_scores(model, get_text(val, best))
majority_pred = DummyClassifier(strategy="most_frequent").fit(train["message_clean"], train["label"]).predict(val["message_clean"])

metrics = {
    "Train": evaluate(train["label"], toxic_scores(model, get_text(train, best)), THRESHOLD),
    "Validation, ngưỡng 0": evaluate(val["label"], val_scores, 0.0),
    "Validation, ngưỡng đã chọn": evaluate(val["label"], val_scores, THRESHOLD),
    "Lớp phổ biến nhất (Validation)": evaluate(val["label"], majority_pred.astype(float), 0.5),
}
metrics_table = pd.DataFrame(metrics).T.round(4)
metrics_table

In [ ]:
plot_results(val["label"], val_scores, THRESHOLD, "Validation")

## 11. Theo bài đăng: ảnh hưởng của việc trùng bài với Train

In [ ]:
def by_post_overlap(frame, scores):
    """3 file chia theo bình luận nên nhiều dòng có bài đăng nằm trong Train; tách riêng để thấy mức ảnh hưởng."""
    seen = frame["post_id"].isin(train_posts).values
    return pd.DataFrame({
        name: {"so_dong": int(mask.sum()), **evaluate(frame["label"].values[mask], scores[mask], THRESHOLD)}
        for name, mask in {"Bài đăng có trong Train": seen, "Bài đăng không có trong Train": ~seen}.items()
    }).T.round(4)


by_post_overlap(val, val_scores)

## 12. Đặc trưng ảnh hưởng nhất

In [ ]:
names = model[:-1].get_feature_names_out()
clf = model.named_steps["clf"]
weights = pd.Series(clf.coef_[0] if hasattr(clf, "coef_") else clf.feature_log_prob_[1] - clf.feature_log_prob_[0], index=names).sort_values()
pd.DataFrame({
    "nghieng_toxic": weights.index[::-1][:20], "trong_so_toxic": weights.values[::-1][:20].round(3),
    "nghieng_non_toxic": weights.index[:20], "trong_so_non_toxic": weights.values[:20].round(3),
})

## 13. Phân tích lỗi trên Validation

In [ ]:
errors = val.assign(score=val_scores, pred=(val_scores >= THRESHOLD).astype(int))
errors = errors[errors["pred"] != errors["label"]]
errors["loai_loi"] = np.where(errors["label"] == 1, "FN: bỏ sót toxic", "FP: báo nhầm toxic")
errors = errors.sort_values("score", key=np.abs, ascending=False)[
    ["sample_id", "post_id", "loai_loi", "score", "post_message", "parent_message", "message_clean"]
]
errors.to_csv(OUT / "val_errors.csv", index=False, encoding="utf-8-sig")
print(errors["loai_loi"].value_counts().to_dict())
errors[["loai_loi", "score", "message_clean"]].head(15)

## 14. Lưu kết quả

In [ ]:
joblib.dump({"model": model, "threshold": THRESHOLD, "config": best}, OUT / "model.joblib")
report = {"experiment": EXPERIMENT, "model": MODEL_NAME, "seed": SEED, "config": best, "threshold": THRESHOLD,
          "cv_f1": float(top.loc[0, "cv_f1"]), "metrics": metrics}
(OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu vào", OUT)

## 15. Đánh giá cuối trên Test

In [ ]:
if RUN_TEST:
    test_scores = toxic_scores(model, get_text(test, best))
    report["metrics"]["Test"] = evaluate(test["label"], test_scores, THRESHOLD)
    test_slices = by_post_overlap(test, test_scores)
    report["test_theo_bai_dang"] = test_slices.to_dict("index")
    (OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    test.assign(score=test_scores, pred=(test_scores >= THRESHOLD).astype(int))[
        ["sample_id", "post_id", "label", "pred", "score", "message_clean"]
    ].to_csv(OUT / "test_predictions.csv", index=False, encoding="utf-8-sig")
    plot_results(test["label"], test_scores, THRESHOLD, "Test")
    display(pd.DataFrame({"Test": report["metrics"]["Test"]}).T.round(4))
    display(test_slices)
else:
    print("Chưa đánh giá Test. Đặt RUN_TEST = True sau khi đã chốt cấu hình, rồi chạy lại toàn bộ notebook.")

## 16. Tải kết quả về máy

In [ ]:
# Máy Colab bị xóa khi hết phiên, nên tải kết quả về máy trước khi đóng
archive = shutil.make_archive(str(BASE / f"ket_qua_{EXPERIMENT}"), "zip", OUT)
if IN_COLAB:
    files.download(archive)
print("Đã đóng gói:", archive)